In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aradhanahirapara/product-retail-price-survey-2017-2025")

print("Path to dataset files:", path)

/home/kimo/instaling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 2.43M/2.43M [00:00<00:00, 3.40MB/s]

Extracting files...


Path to dataset files: /home/kimo/.cache/kagglehub/datasets/aradhanahirapara/product-retail-price-survey-2017-2025/versions/5


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import warnings
warnings.filterwarnings("ignore")

# -------------------------------
# 0. PREPARE DATA (ensure VALUE column)
# -------------------------------
df = pd.read_csv("/content/product-retail-price-survey-2017-2025/Retail_Prices_of _Products.csv")
df = df.copy()
if 'Value' in df.columns:
    df = df.rename(columns={'Value': 'VALUE'})
df['date'] = pd.to_datetime(df['Year'].astype(str) + '-' + df['Month'].astype(str).str.zfill(2) + '-01')
df = df.sort_values(['GEO', 'Product Category', 'Products', 'Essential', 'date']).reset_index(drop=True)
df = df[['GEO', 'Product Category', 'Products', 'Essential', 'date', 'VALUE']].dropna(subset=['VALUE'])

# Encode categoricals
categorical_cols = ['GEO', 'Product Category', 'Products', 'Essential']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col + '_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le

group_keys = ['GEO', 'Product Category', 'Products', 'Essential']
encoded_keys = [col + '_encoded' for col in categorical_cols]

# -------------------------------
# 1. ADVANCED FEATURE ENGINEERING
# -------------------------------
def build_advanced_features(group):
    group = group.set_index('date').sort_index()
    full_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='MS')
    group = group.reindex(full_range)
    
    # Basic lags
    for lag in [1, 2, 3, 4, 6, 12]:
        group[f'lag_{lag}'] = group['VALUE'].shift(lag)
    
    # Same month last year (strong seasonal signal)
    group['lag_12'] = group['VALUE'].shift(12)
    
    # Rolling stats
    group['rolling_mean_3'] = group['VALUE'].shift(1).rolling(3, min_periods=1).mean()
    group['rolling_mean_6'] = group['VALUE'].shift(1).rolling(6, min_periods=1).mean()
    group['rolling_std_3'] = group['VALUE'].shift(1).rolling(3, min_periods=1).std()
    group['rolling_cv_3'] = group['rolling_std_3'] / (group['rolling_mean_3'] + 1e-8)  # coefficient of variation
    
    # First-order difference (momentum)
    group['diff_1'] = group['VALUE'].diff(1)
    group['diff_2'] = group['VALUE'].diff(2)
    
    # Relative changes (%)
    group['pct_change_1'] = group['VALUE'].pct_change(1)
    group['pct_change_3'] = group['VALUE'].pct_change(3)
    
    # Time features
    group['year'] = group.index.year
    group['month'] = group.index.month
    group['quarter'] = group.index.quarter
    group['month_sin'] = np.sin(2 * np.pi * group['month'] / 12)
    group['month_cos'] = np.cos(2 * np.pi * group['month'] / 12)
    
    # Linear & quadratic time trend (per group)
    group['time_idx'] = np.arange(len(group))
    group['time_trend'] = group['time_idx']
    group['time_trend_sq'] = group['time_idx'] ** 2
    
    return group.reset_index().rename(columns={'index': 'date'})

# Apply per group
feature_dfs = []
for name, group in df.groupby(group_keys):
    if len(group) < 12:  # need at least 1 year
        continue
    feat_df = build_advanced_features(group)
    for col in categorical_cols:
        feat_df[col + '_encoded'] = label_encoders[col].transform([name[categorical_cols.index(col)]])[0]
    feature_dfs.append(feat_df)

if not feature_dfs:
    raise ValueError("Not enough data.")
features_df = pd.concat(feature_dfs, ignore_index=True)

# -------------------------------
# 2. PREPARE MODELING DATA
# -------------------------------
# Chronological split: last 6 months = validation
def assign_chrono_split(group):
    n = len(group)
    if n <= 6:
        group['split'] = 'train'
    else:
        group['split'] = 'train'
        group.iloc[-6:, group.columns.get_loc('split')] = 'val'
    return group

features_df = features_df.groupby(group_keys, group_keys=False).apply(assign_chrono_split).reset_index(drop=True)

# Define all feature columns
lag_cols = [f'lag_{i}' for i in [1,2,3,4,6,12]]
rolling_cols = ['rolling_mean_3', 'rolling_mean_6', 'rolling_std_3', 'rolling_cv_3']
diff_cols = ['diff_1', 'diff_2']
pct_cols = ['pct_change_1', 'pct_change_3']
time_cols = ['year', 'month_sin', 'month_cos', 'quarter', 'time_trend', 'time_trend_sq']
feature_cols = lag_cols + rolling_cols + diff_cols + pct_cols + time_cols + encoded_keys

# Clean data
modeling_df = features_df.dropna(subset=['VALUE']).copy()
modeling_df[feature_cols] = modeling_df[feature_cols].fillna(0)

train_df = modeling_df[modeling_df['split'] == 'train']
val_df = modeling_df[modeling_df['split'] == 'val']

X_train = train_df[feature_cols]
y_train = train_df['VALUE']
X_val = val_df[feature_cols]
y_val = val_df['VALUE']

print(f"✅ Features: {len(feature_cols)} | Train: {len(X_train):,} | Val: {len(X_val):,}")

# -------------------------------
# 3. RANDOMIZED SEARCH FOR HYPERPARAMETERS
# -------------------------------
# Define search space
param_dist = {
    'n_estimators': randint(400, 1000),
    'max_depth': randint(6, 12),
    'learning_rate': uniform(0.01, 0.08),  # 0.01 to 0.09
    'subsample': uniform(0.7, 0.25),       # 0.7 to 0.95
    'colsample_bytree': uniform(0.7, 0.25),
    'reg_alpha': uniform(0.1, 1.0),
    'reg_lambda': uniform(0.5, 2.0)
}

# Use a small number of iterations for speed (increase if you have time)
xgb_model = xgb.XGBRegressor(random_state=42, tree_method='hist', eval_metric='rmse')

# Since we can't use standard CV (grouped time series), we'll use **manual validation**
# But RandomizedSearchCV doesn't support grouped time series easily → we do **manual random search**

print("\n🔍 Performing Manual Random Search (5 trials)...")
best_score = float('inf')
best_params = None

np.random.seed(42)
for trial in range(5):  # Increase to 10–20 for better results
    params = {
        'n_estimators': np.random.randint(400, 1001),
        'max_depth': np.random.randint(6, 13),
        'learning_rate': np.random.uniform(0.01, 0.09),
        'subsample': np.clip(np.random.uniform(0.7, 0.95), 0.5, 1.0),
        'colsample_bytree': np.clip(np.random.uniform(0.7, 0.95), 0.5, 1.0),
        'reg_alpha': np.random.uniform(0.1, 1.0),
        'reg_lambda': np.random.uniform(0.5, 2.5),
        'random_state': 42,
        'early_stopping_rounds': 50,
        'eval_metric': 'rmse'
    }
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    
    if rmse < best_score:
        best_score = rmse
        best_params = params
        best_model = model
    print(f"  Trial {trial+1}: RMSE={rmse:.4f} | LR={params['learning_rate']:.3f}, Depth={params['max_depth']}")

print(f"\n🏆 Best RMSE: {best_score:.4f}")
print("Best params:", {k: round(v, 4) if isinstance(v, float) else v for k, v in best_params.items()})

# Use best model
val_pred = best_model.predict(X_val)
val_pred = np.clip(val_pred, y_train.min(), y_train.max() * 2)  # reasonable clip

val_df = val_df.copy()
val_df['predicted_VALUE'] = val_pred

# -------------------------------
# 4. EVALUATION: MAE, RMSE, MAPE, R²
# -------------------------------
mae = mean_absolute_error(y_val, val_pred)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
mape = mean_absolute_percentage_error(y_val, val_pred) * 100
r2 = r2_score(y_val, val_pred)

print("\n" + "="*50)
print("📈 FINAL MODEL PERFORMANCE (VALIDATION)")
print("="*50)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")
print("="*50)

# -------------------------------
# 5. VISUALIZATIONS
# -------------------------------
# Plot actual vs predicted
np.random.seed(42)
sample_groups = val_df[group_keys].drop_duplicates().sample(n=min(6, 100), replace=False)

plt.figure(figsize=(18, 12))
for i, (_, gv) in enumerate(sample_groups.iterrows()):
    full = modeling_df[
        (modeling_df['GEO'] == gv['GEO']) &
        (modeling_df['Product Category'] == gv['Product Category']) &
        (modeling_df['Products'] == gv['Products']) &
        (modeling_df['Essential'] == gv['Essential'])
    ].sort_values('date')
    val_part = val_df[
        (val_df['GEO'] == gv['GEO']) &
        (val_df['Product Category'] == gv['Product Category']) &
        (val_df['Products'] == gv['Products']) &
        (val_df['Essential'] == gv['Essential'])
    ].sort_values('date')
    
    ax = plt.subplot(2, 3, i+1)
    ax.plot(full['date'], full['VALUE'], color='lightgray', linewidth=1)
    ax.plot(val_part['date'], val_part['VALUE'], 'o-', color='blue', label='Actual')
    ax.plot(val_part['date'], val_part['predicted_VALUE'], 'x--', color='red', label='Predicted')
    ax.set_title(f"{gv['GEO']} | {gv['Products'][:20]}...", fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.suptitle('Validation: Actual vs Predicted (Advanced Features + Tuned XGBoost)', y=1.02)
plt.show()

# Residuals
resid = y_val - val_pred
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(val_pred, resid, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted'); plt.ylabel('Residuals'); plt.title('Residuals vs Predicted')
plt.subplot(1, 2, 2)
sns.histplot(resid, kde=True, color='green')
plt.title('Residual Distribution')
plt.tight_layout()
plt.show()

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, y='feature', x='importance', palette='magma')
plt.title('Top 15 Features (Advanced Model)')
plt.tight_layout()
plt.show()

In [ ]:
val_pred = best_model.predict(X_val)
val_pred = np.clip(val_pred, y_train.min(), y_train.max() * 2)  # reasonable clip

val_df = val_df.copy()
val_df['predicted_VALUE'] = val_pred

# -------------------------------
# 4. EVALUATION: MAE, RMSE, MAPE, R²
# -------------------------------
mae = mean_absolute_error(y_val, val_pred)
rmse = np.sqrt(mean_squared_error(y_val, val_pred))
mape = mean_absolute_percentage_error(y_val, val_pred) * 100
r2 = r2_score(y_val, val_pred)

print("\n" + "="*50)
print("📈 FINAL MODEL PERFORMANCE (VALIDATION)")
print("="*50)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")
print("="*50)

# -------------------------------
# 5. VISUALIZATIONS
# -------------------------------
# Plot actual vs predicted
np.random.seed(42)
sample_groups = val_df[group_keys].drop_duplicates().sample(n=min(6, 100), replace=False)

plt.figure(figsize=(18, 12))
for i, (_, gv) in enumerate(sample_groups.iterrows()):
    full = modeling_df[
        (modeling_df['GEO'] == gv['GEO']) &
        (modeling_df['Product Category'] == gv['Product Category']) &
        (modeling_df['Products'] == gv['Products']) &
        (modeling_df['Essential'] == gv['Essential'])
    ].sort_values('date')
    val_part = val_df[
        (val_df['GEO'] == gv['GEO']) &
        (val_df['Product Category'] == gv['Product Category']) &
        (val_df['Products'] == gv['Products']) &
        (val_df['Essential'] == gv['Essential'])
    ].sort_values('date')
    
    ax = plt.subplot(2, 3, i+1)
    ax.plot(full['date'], full['VALUE'], color='lightgray', linewidth=1)
    ax.plot(val_part['date'], val_part['VALUE'], 'o-', color='blue', label='Actual')
    ax.plot(val_part['date'], val_part['predicted_VALUE'], 'x--', color='red', label='Predicted')
    ax.set_title(f"{gv['GEO']} | {gv['Products'][:20]}...", fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.suptitle('Validation: Actual vs Predicted (Advanced Features + Tuned XGBoost)', y=1.02)
plt.show()

# Residuals
resid = y_val - val_pred
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(val_pred, resid, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted'); plt.ylabel('Residuals'); plt.title('Residuals vs Predicted')
plt.subplot(1, 2, 2)
sns.histplot(resid, kde=True, color='green')
plt.title('Residual Distribution')
plt.tight_layout()
plt.show()

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, y='feature', x='importance', palette='magma')
plt.title('Top 15 Features (Advanced Model)')
plt.tight_layout()
plt.show()